In [81]:
from collections import defaultdict , Counter
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import pandas as pd
from sklearn.model_selection import train_test_split

In [94]:
df = pd.read_csv('hmm_tag_dataset_1.csv')
df.head()

,sentence
0,Mary/NNP Jane/NNP can/MD see/VB Will/NNP
1,Spot/NNP will/MD see/VB Mary/NNP
2,Will/MD Jane/NNP spot/VB Mary/NNP
3,Mary/NNP will/MD pat/VB Spot/NNP
4,Will/NNP can/MD spot/VB Mary/NNP


In [95]:
data = []
for sent in df["sentence"]:
    pairs = []
    for tokens in sent.split():
        word,tag = tokens.rsplit('/',1)
        pairs.append((word,tag))
    data.append(pairs)

data

[[('Mary', 'NNP'),
  ('Jane', 'NNP'),
  ('can', 'MD'),
  ('see', 'VB'),
  ('Will', 'NNP')],
 [('Spot', 'NNP'), ('will', 'MD'), ('see', 'VB'), ('Mary', 'NNP')],
 [('Will', 'MD'), ('Jane', 'NNP'), ('spot', 'VB'), ('Mary', 'NNP')],
 [('Mary', 'NNP'), ('will', 'MD'), ('pat', 'VB'), ('Spot', 'NNP')],
 [('Will', 'NNP'), ('can', 'MD'), ('spot', 'VB'), ('Mary', 'NNP')],
 [('Mary', 'NNP'), ('will', 'MD'), ('see', 'VB'), ('Spot', 'NNP')],
 [('Mary', 'NNP'), ('can', 'MD'), ('see', 'VB')],
 [('Jane', 'NNP'), ('will', 'MD'), ('pat', 'VB'), ('Spot', 'NNP')],
 [('Spot', 'NNP'), ('see', 'VB'), ('Mary', 'NNP')]]

In [96]:
train_sentences,test_sentences = train_test_split(data,test_size = 0.2,random_state = 42)

In [97]:
# train_sentences = [
#     [("Mary","NNP"),("Jane","NNP"),("can","MD"),("see","VB"),("Will","NNP")],
#     [("Spot","NNP"),("will","MD"),("see","VB"),("Mary","NNP")],
#     [("Will","MD"),("Jane","NNP"),("spot","VB"),("Mary","NNP")],
#     [("Mary","NNP"),("will","MD"),("pat","VB"),("Spot","NNP")]
# ]

# test_sentences = [
#     [("Will","NNP"),("can","MD"),("spot","VB"),("Mary","NNP")],
#     [("Mary","NNP"),("will","MD"),("see","VB"),("Spot","NNP")],
#     [("Mary","NNP"),("can","MD"),("see","VB")],
#     [("Jane","NNP"),("will","MD"),("pat","VB"),("Spot","NNP")],
#     [("Spot","NNP"),("see","VB"),("Mary","NNP")]
# ]

In [98]:
tag_count = Counter()
trans_count = defaultdict(Counter)
emiss_count = defaultdict(Counter)
vocab = set()
tags = set()

In [ ]:
for sentence in train_sentences:
    prev = '<s>'
    for word,tag in sentence:
        tag_count[tag] += 1
        trans_count[prev][tag] += 1
        emiss_count[tag][word.lower()] += 1
        vocab.add(word.lower())
        tags.add(tag) 
        prev = tag
    trans_count[prev]['</s>'] += 1


trans_count

defaultdict(collections.Counter,
            {'<s>': Counter({'NNP': 6, 'MD': 1}),
             'NNP': Counter({'</s>': 6, 'MD': 5, 'VB': 2, 'NNP': 1}),
             'MD': Counter({'VB': 5, 'NNP': 1}),
             'VB': Counter({'NNP': 6, '</s>': 1})})

In [100]:
initial_prob = {}
total_starts = sum(trans_count['<s>'].values())
for tag in tags:
    initial_prob[tag] = trans_count['<s>'].get(tag,0) / total_starts

initial_prob

{'MD': 0.14285714285714285, 'NNP': 0.8571428571428571, 'VB': 0.0}

In [ ]:
trans_prob = {}
for prev in set(trans_count) | tags:
    total_trans = sum(trans_count[prev].values()) + len(tags)
    trans_prob[prev] = {} 
    for tag in tags:
        count = trans_count[prev].get(tag,0)
        trans_prob[prev][tag] = (count + 1)/total_trans

emiss_prob = {}
for tag in tags:
    total_emiss = sum(emiss_count[tag].values()) + len(vocab)
    emiss_prob[tag] = {}
    for word in vocab:
        count = emiss_count[tag].get(word.lower(),0)
        emiss_prob[tag][word] = (count + 1)/total_emiss

In [102]:
unk_prob = 1/(len(vocab)*10)
tag_list = list(tags)

In [ ]:
def viterbi(words,trans_prob,emiss_prob,initial_prob,unk_prob,tags):
    n = len(words)
    V = [{} for _ in range(n)]
    backpointer = [{} for _ in range(n)]

    for tag in tags:
        emission = emiss_prob[tag].get(words[0].lower(),unk_prob)
        V[0][tag] = initial_prob.get(tag,1e-6)*emission 
        backpointer[0][tag] = None

    for t in range(1,n):
        for tag in tags:
            emission = emiss_prob[tag].get(words[t].lower(), unk_prob)
            prob = {}
            for prev in tags:
                prob[prev] = V[t-1][prev] * trans_prob[prev][tag] * emission
            best_prev = max(prob,key = prob.get)
            V[t][tag] = prob[best_prev]
            backpointer[t][tag] = best_prev

    best_last_tag = max(V[n-1],key = V[n-1].get)
    path = [None]*n
    path[n-1] = best_last_tag
    for t in range(n-2,-1,-1):
        path[t] = backpointer[t+1][path[t+1]]

    return path


In [104]:
def evaluation(test_sentences,initial_prob,trans_prob,emiss_prob,unk_prob,tags):
    y_true = []
    y_pred = []

    for i,sent in enumerate(test_sentences,1):
        words = [w for w,_ in sent]
        gold_tags = [t for _,t in sent]
        predicted_tags = viterbi(words,trans_prob,emiss_prob,initial_prob,unk_prob,tags)
        correct = gold_tags == predicted_tags
        print("sentence {i}:")
        print("GOLD TAGS : ",gold_tags)
        print("PREDICTED TAGS : ",predicted_tags)
        if correct:
            print("Correct")
        else:
            print("Incorrect")
        print('*'*30)

        y_true.extend(gold_tags)
        y_pred.extend(predicted_tags)

    acc = accuracy_score(y_true,y_pred)
    print("Accuracy : ", acc)

    print("CLASSIFICATION REPORT")
    class_report = classification_report(y_true,y_pred,digits = 4)
    print(class_report)

    print("CONFUSION MATRIX")
    labels = sorted(tags)
    conf_matrix = confusion_matrix(y_true,y_pred,labels=labels)
    print(labels)
    print(conf_matrix)

In [105]:
evaluation(test_sentences,initial_prob,trans_prob,emiss_prob,unk_prob,tag_list)

sentence {i}:
GOLD TAGS :  ['NNP', 'MD', 'VB', 'NNP']
PREDICTED TAGS :  ['NNP', 'MD', 'VB', 'NNP']
Correct
******************************
sentence {i}:
GOLD TAGS :  ['NNP', 'MD', 'VB', 'NNP']
PREDICTED TAGS :  ['NNP', 'MD', 'VB', 'NNP']
Correct
******************************
Accuracy :  1.0
CLASSIFICATION REPORT
              precision    recall  f1-score   support

          MD     1.0000    1.0000    1.0000         2
         NNP     1.0000    1.0000    1.0000         4
          VB     1.0000    1.0000    1.0000         2

    accuracy                         1.0000         8
   macro avg     1.0000    1.0000    1.0000         8
weighted avg     1.0000    1.0000    1.0000         8

CONFUSION MATRIX
['MD', 'NNP', 'VB']
[[2 0 0]
 [0 4 0]
 [0 0 2]]
